# Clustering Documents with K-Means

**Computational Text Analysis Pack · Module 06 — Clustering Documents**

*Technique:* K-means on TF-IDF; choosing k and reading clusters honestly

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/05-unsupervised-structure/06b_clustering_documents.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

K-means groups documents that use similar words, without being told what the
groups are. It is the same algorithm you ran on housing data; only the columns
have changed.

This corpus has a rare advantage: it carries **topic labels**. So you can cluster
without them and then check the clusters against them — a validity test almost no
real DH project can run. Use it to calibrate how much to trust clustering when
you have no labels.

## The tooling

**scikit-learn** — `TfidfVectorizer` for the document-term matrix, `KMeans` and
`TruncatedSVD` for structure. **scipy**'s `cluster.hierarchy` does the
dendrograms. **gensim** is the alternative once the matrix stops fitting in
memory.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

LANGUAGE = "English"        # ← change this
K = 5                       # ← number of clusters

sub = df[df["language"] == LANGUAGE].reset_index(drop=True)
vec = TfidfVectorizer(token_pattern=r"\w+", min_df=5, max_features=3000,
                      sublinear_tf=True)
X = vec.fit_transform(sub["text"])
vocab = np.array(vec.get_feature_names_out())

km = KMeans(n_clusters=K, n_init=10, random_state=0)
sub["cluster"] = km.fit_predict(X)
print(sub["cluster"].value_counts().sort_index().to_string())

## Naming a cluster

A cluster number means nothing. What makes it interpretable is its **centroid** —
the average document — and the words with the highest weight in it.

In [ ]:
centroids = km.cluster_centers_
for c in range(K):
    top = vocab[np.argsort(centroids[c])[::-1][:10]]
    print(f"cluster {c} (n={(sub['cluster'] == c).sum():>4}): {', '.join(top)}")

In [ ]:
# The validity check the labels make possible.
ct = pd.crosstab(sub["cluster"], sub["category"])
print(ct.to_string())
print()
pct = ct.div(ct.sum(axis=1), axis=0) * 100
print("row percentages — how pure is each cluster?")
print(pct.to_string(float_format=lambda v: f"{v:.0f}"))

In [ ]:
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(sub["category"], sub["cluster"])
print(f"adjusted Rand index vs the topic labels: {ari:.3f}")
print()
print("0 = no better than random, 1 = perfect agreement. Typical values for")
print("text clustering are modest — and that is the honest baseline for what")
print("clustering can recover when you have no labels to check against.")

### 🔧 Your turn 1

Change `K` to 3, then to 10, and re-run.

Watch what happens to the ARI and to the cluster word lists. More clusters
usually means more interpretable small groups and a *lower* score against the
coarse labels — the algorithm is finding real structure the labels do not encode.

## Choosing k

Two standard aids, neither of which gives you an answer.

In [ ]:
from sklearn.metrics import silhouette_score

ks = [2, 3, 4, 5, 6, 8, 10, 12]
inertia, sil = [], []
for k in ks:
    m = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    inertia.append(m.inertia_)
    sil.append(silhouette_score(X, m.labels_, sample_size=min(2000, X.shape[0]),
                                random_state=0))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(ks, inertia, marker="o")
axes[0].set_xlabel("k"); axes[0].set_ylabel("inertia")
axes[0].set_title("elbow plot — look for a bend")
axes[1].plot(ks, sil, marker="o", color="seagreen")
axes[1].set_xlabel("k"); axes[1].set_ylabel("silhouette")
axes[1].set_title("silhouette — higher is more separated")
plt.tight_layout(); plt.show()

print("Text data rarely gives a clean elbow, and silhouette scores are low")
print("across the board because documents overlap in vocabulary. Neither plot")
print("will choose k for you. Interpretability usually should.")

In [ ]:
# A view of the space. Two dimensions out of three thousand — treat it as
# an illustration, never as evidence that the clusters are separate.
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=2, random_state=0)
coords = svd.fit_transform(X)
fig, ax = plt.subplots(figsize=(8, 6))
for c in range(K):
    m = sub["cluster"] == c
    ax.scatter(coords[m, 0], coords[m, 1], s=8, alpha=0.5, label=f"cluster {c}")
ax.set_xlabel(f"component 1 ({svd.explained_variance_ratio_[0]:.1%} of variance)")
ax.set_ylabel(f"component 2 ({svd.explained_variance_ratio_[1]:.1%})")
ax.set_title("Documents projected to two dimensions")
ax.legend(markerscale=2)
plt.tight_layout(); plt.show()
print(f"These two components carry {svd.explained_variance_ratio_.sum():.1%} of")
print("the variance. Clusters that overlap here may be well separated in the")
print("dimensions you cannot see — and vice versa.")

### 🔧 Your turn 2

Read three documents from one cluster.

Do they belong together for the reason the centroid words suggest? If a cluster
turns out to be "articles that mention a particular agency", that is still a real
finding — but it is not the theme you would have written on the chart.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** ARI usually peaks near the true number of categories and falls
away in both directions. At k=10 the clusters often become more coherent to read
while scoring worse against the labels, because they split *politics* into
elections, courts and parliament. Neither the score nor your reading is the
authority on its own; disagreement between them is information.

**Your turn 2.** Very often a cluster is held together by something other than
topic — an outlet's boilerplate, a recurring byline, a standard closing formula.
This is the standard failure mode of clustering news text and the reason to read
documents before naming clusters. If boilerplate is driving it, remove those
strings and re-run, and say in your methods that you did.

</details>

## Where this stops

You can group documents and check the groups. K-means forces every document into
exactly one group and needs k in advance. Next: **06c — hierarchical clustering**,
which does neither.